# Part 2: Job Postings Analysis - Role Categorization and Requirements Extraction

This notebook uses **LangChain** with a **local Ollama model (`llama3.2`, a small language model)** to analyze job postings and produce, for each posting:

1. **Job Category Classification** — a broad domain (Technology/IT, Finance, Marketing, Healthcare, Education, etc.)
2. **Key Requirements Extraction** — required skills/technologies, education level, and experience

As in Part 1, we use Ollama rather than a hosted API to avoid rate limits and to make the full-dataset bonus feasible.

**Name:** Aishwarya Nevrekar  
**Registration No.:** 27PGAI0028

In [1]:
import json
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from typing import List, Literal
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

MODEL_NAME = "llama3.2"  # local SLM served via Ollama
OLLAMA_URL = "http://localhost:11434"  # explicit address: an OLLAMA_HOST of 0.0.0.0 is not a valid client address
# num_predict caps each reply so a looping generation cannot stall a long run
llm = ChatOllama(model=MODEL_NAME, temperature=0, base_url=OLLAMA_URL, num_predict=512)

print(f"Using Ollama model: {MODEL_NAME}")

Using Ollama model: llama3.2


## Step 1: Load the Dataset

Job postings dataset with `Job Title` and `Job Description` columns, scraped from online job boards.

In [2]:
raw_df = pd.read_csv("data/job_title_des.csv", index_col=0)
print(raw_df.shape)
raw_df.head(3)

(2277, 2)


,Job Title,Job Description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ..."


In [3]:
df = raw_df.rename(columns={"Job Title": "Job_Title", "Job Description": "Job_Description"}).copy()
df = df.head(25).reset_index(drop=True)
print(df.shape)
df.head()

(25, 2)


,Job_Title,Job_Description
0,Flutter Developer,We are looking for hire experts flutter develo...
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ..."
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...
4,Full Stack Developer,job responsibility full stack engineer – react...


## Step 2: Job Category Classification

A prompt (with a few labeled examples) that maps a job title + description to a broad domain category, using `with_structured_output` for a clean single label. "Other" is used as a fallback when the role doesn't clearly fit.

In [4]:
CATEGORIES = [
    "Technology/IT", "Finance", "Marketing", "Healthcare", "Education",
    "Sales", "Human Resources", "Operations", "Design", "Customer Service", "Other",
]

class JobCategory(BaseModel):
    category: Literal[tuple(CATEGORIES)] = Field(description="The single best-fitting broad domain for this job")

classification_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given a job title and description, categorize the job into one of the following domains: "
     f"{', '.join(CATEGORIES)}. If unsure, use 'Other'. Respond with only the single best-fitting category.\n\n"
     "Examples:\n"
     "Job: Software Engineer. Description: Build and maintain backend services using Java and Spring Boot.\n"
     "Domain category: Technology/IT\n\n"
     "Job: Staff Accountant. Description: Prepare financial statements and reconcile accounts monthly.\n"
     "Domain category: Finance\n\n"
     "Job: Registered Nurse. Description: Provide patient care and administer medication in a hospital ward.\n"
     "Domain category: Healthcare"),
    ("human", "Job: {title}. Description: {description}\n\nDomain category:")
])

classification_chain = classification_prompt | llm.with_structured_output(JobCategory)

# --- Test on a sample datapoint ---
sample = df.iloc[0]
sample_result = classification_chain.invoke({"title": sample["Job_Title"], "description": sample["Job_Description"][:3000]})
print("Job Title:", sample["Job_Title"])
print("Predicted category:", sample_result.category)

Job Title: Flutter Developer
Predicted category: Technology/IT


## Step 3: Key Requirements Extraction

A single composite prompt extracts **Skills**, **Education**, and **Experience** in one call, using **JSON-mode prompting** (Ollama's `format="json"`) with a Pydantic schema validated via `JsonOutputParser`. In testing, LangChain's tool-calling `with_structured_output` frequently returned empty/blank fields on longer descriptions with this small model — JSON-mode prompting proved much more reliable. Missing information is explicitly represented as `"Not specified"` rather than left to guesswork.

In [5]:
class JobRequirements(BaseModel):
    skills: List[str] = Field(default_factory=list, description="Key skills, technologies, or tools mentioned; empty list if none")
    education: str = Field(description="Minimum/preferred education level, e.g. Bachelor's degree in CS; use 'Not specified' if not mentioned")
    experience: str = Field(description="Years of experience or experience level required, e.g. '3+ years'; use 'Not specified' if not mentioned")

requirements_parser = JsonOutputParser(pydantic_object=JobRequirements)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Extract the required skills/technologies, minimum education level, and required years of experience "
     "from the job description below. If a piece of information is not mentioned, use 'Not specified' "
     "(for education/experience) or an empty list (for skills). Do not invent information.\n{format_instructions}"),
    ("human", "Job Title: {title}\nJob Description:\n{description}")
]).partial(format_instructions=requirements_parser.get_format_instructions())

# JSON-mode (format="json") makes the Ollama model emit valid JSON directly, which this small
# model handles more reliably than LangChain's tool-calling with_structured_output.
requirements_llm = ChatOllama(model=MODEL_NAME, temperature=0, format="json", base_url=OLLAMA_URL, num_predict=512)
extraction_chain = extraction_prompt | requirements_llm | requirements_parser

# --- Test on a sample datapoint ---
sample_reqs = extraction_chain.invoke({"title": sample["Job_Title"], "description": sample["Job_Description"][:8000]})
print("Job Title:", sample["Job_Title"])
print("Requirements:", sample_reqs)

Job Title: Flutter Developer
Requirements: {'skills': [], 'education': 'Not specified', 'experience': 'Not specified'}


## Step 4 & 5: Apply to All Postings and Update the DataFrame

In [6]:
def analyze_posting(title: str, description: str) -> dict:
    desc = str(description)[:8000]  # requirements are often listed near the end of long postings
    try:
        category = classification_chain.invoke({"title": title, "description": desc[:3000]}).category
    except Exception:
        category = "Other"
    try:
        reqs = extraction_chain.invoke({"title": title, "description": desc})
        skills = reqs.get("skills") or []
        if isinstance(skills, str):  # e.g. "Python, SQL" instead of a list
            skills = [s.strip() for s in skills.split(",") if s.strip()]
        education = reqs.get("education") or "Not specified"
        experience = reqs.get("experience") or "Not specified"
        if isinstance(education, list):
            education = ", ".join(map(str, education))
        if isinstance(experience, list):
            experience = ", ".join(map(str, experience))
    except Exception:
        skills, education, experience = [], "Not specified", "Not specified"
    return {
        "Predicted_Category": category,
        "Required_Skills": skills,
        "Education_Required": education,
        "Experience_Required": experience,
    }


def run_pipeline(data: pd.DataFrame, cache_path: str | None = None, workers: int = 1) -> pd.DataFrame:
    """Analyze every posting. With `cache_path`, each finished row is saved to a JSON-lines file so a
    long run (the bonus) can resume where it stopped. Rows with nothing extracted are retried once.
    `workers` > 1 sends several postings to Ollama at once (it serves requests in parallel)."""
    done = {}
    if cache_path and Path(cache_path).exists():
        for line in Path(cache_path).read_text(encoding="utf-8").splitlines():
            record = json.loads(line)
            done[record["row"]] = record["result"]
    todo = [(int(idx), row["Job_Title"], row["Job_Description"]) for idx, row in data.iterrows()
            if int(idx) not in done]
    cache = open(cache_path, "a", encoding="utf-8") if cache_path else None
    lock = threading.Lock()

    def work(item):
        key, title, description = item
        result = analyze_posting(title, description)
        if not result["Required_Skills"] and result["Education_Required"] == result["Experience_Required"] == "Not specified":
            result = analyze_posting(title, description)  # retry once
        with lock:
            done[key] = result
            if cache:
                cache.write(json.dumps({"row": key, "result": result}) + "\n")
                cache.flush()

    with ThreadPoolExecutor(max_workers=workers) as pool:
        list(tqdm(pool.map(work, todo), total=len(todo), desc="Analyzing job postings"))
    if cache:
        cache.close()
    results = [done[int(k)] for k in data.index]
    results_df = pd.DataFrame(results)
    return pd.concat([data.reset_index(drop=True), results_df], axis=1)


final_df = run_pipeline(df)
final_df.head()

Analyzing job postings:   0%|          | 0/25 [00:00<?, ?it/s]

,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,We are looking for hire experts flutter develo...,Technology/IT,[],Not specified,Not specified
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,Technology/IT,"[Python, Django, API development, REST/RPC, Li...",Not specified,Not specified
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ...",Technology/IT,"[Python, Java, Machine Learning, Deep Learning...",Not specified,Not specified
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...,Technology/IT,"[Objective-C, Cocoa Touch, Core Data, Core Ani...",Not specified,Not specified
4,Full Stack Developer,job responsibility full stack engineer – react...,Technology/IT,"[React, J Native, JavaScript, HTML, CSS, Redux...",Not specified,Not specified


In [7]:
final_df.to_csv("outputs/part2_jobs_first25_results.csv", index=False)
final_df.to_json("outputs/part2_jobs_first25_results.json", orient="records", indent=2)
print("Saved to outputs/part2_jobs_first25_results.csv and .json")
final_df

Saved to outputs/part2_jobs_first25_results.csv and .json


,Job_Title,Job_Description,Predicted_Category,Required_Skills,Education_Required,Experience_Required
0,Flutter Developer,We are looking for hire experts flutter develo...,Technology/IT,[],Not specified,Not specified
1,Django Developer,PYTHON/DJANGO (Developer/Lead) - Job Code(PDJ ...,Technology/IT,"[Python, Django, API development, REST/RPC, Li...",Not specified,Not specified
2,Machine Learning,"Data Scientist (Contractor)\r\n\r\nBangalore, ...",Technology/IT,"[Python, Java, Machine Learning, Deep Learning...",Not specified,Not specified
3,iOS Developer,JOB DESCRIPTION:\r\n\r\nStrong framework outsi...,Technology/IT,"[Objective-C, Cocoa Touch, Core Data, Core Ani...",Not specified,Not specified
4,Full Stack Developer,job responsibility full stack engineer – react...,Technology/IT,"[React, J Native, JavaScript, HTML, CSS, Redux...",Not specified,Not specified
5,Java Developer,Software Developer - Integration*\r\nImmediate...,Technology/IT,"[Proven technical expertise in the design, dev...","Bachelor's Degree in Computer Science, Informa...",2 years
6,Full Stack Developer,senior full stack developer \- 1800026h cwt lo...,Technology/IT,"[NodeJS, Java, MongoDB, Elasticsearch, Redis, ...",B.Sc,2 years
7,JavaScript Developer,"Job Description:\r\n\r\nReactJS + NodeJs, Azur...",Technology/IT,"[ReactJS, NodeJS, Azure Functions, GraphQL]",Not specified,3 - 8 years
8,DevOps Engineer,Main Responsibilities and Deliverables:\r\nMan...,Technology/IT,"[Bash, Ruby, Python, Java, Puppet, Chef, Cloud...",Not specified,Not specified
9,Software Engineer,"Overview\r\n\r\n\r\nBased in Silicon Valley, T...",Technology/IT,"[REST API, C/C++, Python, Go, Git, Gerrit, Jen...","BS or MS; computer engineering, computer scien...",Minimum 7 years of software development experi...


## Verify the Outputs
Spot-check a few postings against their description text, then count how often each field was extracted across all 25 postings.

In [8]:
for _, row in final_df.sample(3, random_state=1).iterrows():
    print(f"--- {row['Job_Title']} | category: {row['Predicted_Category']}")
    print("Description excerpt:", " ".join(str(row["Job_Description"]).split())[:350], "...")
    print("Required skills    :", row["Required_Skills"])
    print("Education required :", row["Education_Required"])
    print("Experience required:", row["Experience_Required"], "\n")

total = len(final_df)
print(f"Skills extracted     : {(final_df['Required_Skills'].apply(len) > 0).sum()}/{total} postings")
print(f"Education mentioned  : {(final_df['Education_Required'] != 'Not specified').sum()}/{total} postings")
print(f"Experience mentioned : {(final_df['Experience_Required'] != 'Not specified').sum()}/{total} postings")

--- Software Engineer | category: Technology/IT
Description excerpt: JOB DESCRIPTION Job Title: Software Engineer I SUMMARY The Software Engineer I is responsible to design, code, and/or configure solutions for moderate complexity Agile stories, as well as writing automated unit and integration-level tests. ESSENTIAL JOB FUNCTIONS/RESPONSIBILITIES Designs, codes, and/or configures solutions for moderate complexity A ...
Required skills    : ['Object-oriented design', 'Java', '.NET development', 'Relational OLTP queries', 'Relational database design', 'XML/XSLT document design', 'JavaScript development', 'HTML5', 'CSS']
Education required : Bachelor's degree or higher education level, or its foreign equivalent, in Computer Science, Computer Information Sciences, and/or related field
Experience required: 6+ years (software development), 4 years minimum, Product Experience: 2 years minimum, 4+ years preferred, Domain Experience: 2 years minimum, 4+ years preferred 

--- Software Engineer 

## Bonus: Full Dataset (all job postings)

`RUN_BONUS = True` below processes every posting in the dataset instead of just the first 25. This runs locally via Ollama with no external rate limits, but will take a while (roughly 1-2 hours depending on hardware, since each posting needs 2 LLM calls: classification and requirements extraction).

Finished rows are saved to `outputs/part2_jobs_ALL_cache.jsonl` as the run goes, so an interrupted run continues where it stopped. Set `RUN_BONUS = False` to skip this cell.

In [9]:
RUN_BONUS = True

if RUN_BONUS:
    bonus_df = raw_df.rename(columns={"Job Title": "Job_Title", "Job Description": "Job_Description"}).copy()

    bonus_results_df = run_pipeline(bonus_df, cache_path="outputs/part2_jobs_ALL_cache.jsonl", workers=4)
    bonus_results_df.to_csv("outputs/part2_jobs_ALL_results.csv", index=False)
    bonus_results_df.to_json("outputs/part2_jobs_ALL_results.json", orient="records", indent=2)
    no_skills = (bonus_results_df["Required_Skills"].apply(len) == 0).sum()
    print(f"Rows analysed: {len(bonus_results_df)} | postings with no skills extracted: {no_skills}")
    print("Saved to outputs/part2_jobs_ALL_results.csv and .json")
else:
    print("RUN_BONUS is False - skipping full-dataset run. Set RUN_BONUS = True to process every posting.")

Analyzing job postings:   0%|          | 0/2277 [00:00<?, ?it/s]

Rows analysed: 2277 | postings with no skills extracted: 51
Saved to outputs/part2_jobs_ALL_results.csv and .json
